# Notebook 03: Alpha Diversity Analysis

Computes within-sample diversity (Shannon entropy, Observed OTUs, Simpson index) across
Colorectal, Breast, and Prostate cancer microbiome datasets.
Statistical comparisons use Kruskal-Wallis with pairwise Mann-Whitney U (Bonferroni corrected).

In [ ]:
# Cell 1 — Imports
import pandas as pd          # pandas: the main tool for working with tables of data (like Excel in Python)
import numpy as np            # numpy: fast math on lists of numbers (averages, logs, sums)
import matplotlib             # matplotlib: the main drawing library for Python charts
matplotlib.use('Agg')         # 'Agg' means save charts to files instead of popping up windows
import matplotlib.pyplot as plt  # pyplot: the easy-to-use part of matplotlib for making plots
import seaborn as sns         # seaborn: makes prettier, statistics-ready charts on top of matplotlib
from scipy import stats       # scipy.stats: ready-made statistical tests (Kruskal-Wallis, Mann-Whitney U)
import warnings               # warnings: controls whether Python shows warning messages
warnings.filterwarnings('ignore')  # hide harmless warning messages so output stays clean
import os                     # os: tools for working with files and folder paths

print("Imports complete.")  # confirm all libraries loaded successfully

In [ ]:
# Cell 2 — Paths and load data
BASE_DIR = r"c:\MyProjects\Project-Proposal\PrivateCoach\In Progress\Project-BioInformatics\Projects\cancer-microbiome\Final_Solution"
RESULTS_DIR = os.path.join(BASE_DIR, "Results")   # folder where processed CSV files are stored
FIGURES_DIR = os.path.join(BASE_DIR, "Figures")   # folder where charts will be saved

os.makedirs(RESULTS_DIR, exist_ok=True)  # create Results folder if it doesn't already exist
os.makedirs(FIGURES_DIR, exist_ok=True)  # create Figures folder if it doesn't already exist

# Load genus-level relative abundance table (rows = samples, columns = bacterial genera, values 0–1)
# Think of it as a spreadsheet: each row is one person's sample, each column is how much of one bacterium they have
abund = pd.read_csv(os.path.join(RESULTS_DIR, "abund_combined_genus.csv"), index_col=0)

# Load metadata table — tells us which cancer type and condition (Cancer/Healthy) each sample belongs to
meta = pd.read_csv(os.path.join(RESULTS_DIR, "meta_combined.csv"), index_col=0)

# Keep only samples that appear in BOTH the abundance table AND the metadata
# (like taking only the rows that match between two spreadsheets)
common_idx = abund.index.intersection(meta.index)  # find overlapping sample IDs
abund = abund.loc[common_idx]   # keep only matching rows in abundance table
meta  = meta.loc[common_idx]    # keep only matching rows in metadata

print(f"Abundance matrix shape : {abund.shape}  (samples x genera)")  # show table size
print(f"Metadata shape         : {meta.shape}")                        # show metadata size
print(f"Cancer types           : {meta['cancer_type'].value_counts().to_dict()}")  # samples per cancer type
print(f"Conditions             : {meta['condition'].value_counts().to_dict()}")    # Cancer vs Healthy counts

In [ ]:
# Cell 3 — Define diversity functions
# Alpha diversity measures how many different bacteria live in ONE sample
# Think of it like measuring how many different species live in a single pond

def shannon_entropy(row):
    """Shannon entropy H = -sum(p * log(p)) for p > 0.
    High value = many different bacteria in roughly equal amounts (diverse gut)
    Low value  = one or two bacteria dominate (less diverse gut)
    """
    row = row[row > 0]          # ignore bacteria with zero abundance (they don't contribute)
    return -np.sum(row * np.log(row))  # formula: H = -Σ(p × ln(p)) — the more even the spread, the higher H

def observed_otus(row):
    """Count of genera with relative abundance > 0.
    The simplest diversity measure: just count how many different bacteria are present at all.
    OTU = Operational Taxonomic Unit, a stand-in for 'species/genus'
    """
    return (row > 0).sum()  # count columns where abundance is greater than zero

def simpson_index(row):
    """Simpson diversity index D = 1 - sum(p^2).
    Measures the probability that two randomly picked bacteria are different species.
    D near 1 = very diverse; D near 0 = one bacterium dominates everything
    """
    row = row[row > 0]          # ignore absent bacteria
    return 1 - np.sum(row ** 2) # formula: D = 1 - Σ(p²) — squaring penalizes dominant bacteria

print("Diversity functions defined: shannon_entropy, observed_otus, simpson_index")

In [ ]:
# Cell 4 — Calculate diversity metrics
# Apply each diversity function to every row (every sample) in the abundance table
# axis=1 means "go row by row" — each row is one patient's microbiome sample
shannon_vals   = abund.apply(shannon_entropy, axis=1)  # calculate Shannon entropy for every sample
observed_vals  = abund.apply(observed_otus,   axis=1)  # count how many genera each sample has
simpson_vals   = abund.apply(simpson_index,   axis=1)  # calculate Simpson index for every sample

# Combine all three diversity metrics plus the metadata labels into one tidy table
diversity_df = pd.DataFrame({
    'shannon':       shannon_vals,   # richness + evenness combined
    'observed_otus': observed_vals,  # simple count of bacteria present
    'simpson':       simpson_vals,   # probability that two random bacteria differ
    'cancer_type':   meta['cancer_type'],  # Colorectal, Breast, or Prostate
    'condition':     meta['condition'],    # Cancer or Healthy
})

print("Diversity metrics calculated.")  # confirm calculation finished
# Show statistics (count, mean, std, min, max) broken down by cancer type
print(diversity_df.groupby('cancer_type')[['shannon', 'observed_otus', 'simpson']].describe().round(3))

In [ ]:
# Cell 5 — Statistical testing
# Goal: find out if the three cancer types have SIGNIFICANTLY DIFFERENT diversity levels
# We use non-parametric tests because microbiome data is rarely normally distributed (not bell-curve shaped)
from itertools import combinations  # tool for generating all possible pairs: (A,B), (A,C), (B,C)

METRICS      = ['shannon', 'observed_otus', 'simpson']  # the three diversity measures to test
CANCER_TYPES = sorted(diversity_df['cancer_type'].unique())  # ['Breast', 'Colorectal', 'Prostate']

stat_rows = []  # will collect results from every statistical test here

for metric in METRICS:
    # Extract the values for each cancer type as separate arrays (needed for statistical tests)
    groups = [
        diversity_df.loc[diversity_df['cancer_type'] == ct, metric].dropna().values
        for ct in CANCER_TYPES
    ]

    # Kruskal-Wallis test: like ANOVA but for non-normal data — asks "are any groups different?"
    # It ranks all values together and checks if one group's ranks are too high/low to be random
    kw_stat, kw_p = stats.kruskal(*groups)  # *groups unpacks the list into separate arguments
    stat_rows.append({
        'metric':      metric,
        'test':        'Kruskal-Wallis',
        'comparison':  'All groups',
        'statistic':   round(kw_stat, 4),
        'p_value':     round(kw_p, 6),
        'significant': kw_p < 0.05,  # True if p < 0.05, meaning unlikely to be random chance
    })
    print(f"\n{metric.upper()} — Kruskal-Wallis: H={kw_stat:.3f}, p={kw_p:.4f} {'*' if kw_p < 0.05 else ''}")

    # If KW is significant, do pairwise Mann-Whitney U tests to find WHICH pairs differ
    # Mann-Whitney U: like a t-test for non-normal data — compares two groups at a time
    if kw_p < 0.05:
        pairs = list(combinations(range(len(CANCER_TYPES)), 2))  # all 3 possible pairs of indices
        n_comparisons = len(pairs)  # = 3 (Breast-Colorectal, Breast-Prostate, Colorectal-Prostate)
        for i, j in pairs:
            u_stat, u_p = stats.mannwhitneyu(groups[i], groups[j], alternative='two-sided')
            # Bonferroni correction: multiply p by number of tests to control false positives
            # (if you run 3 tests, you'd expect ~1 false positive at p<0.05 by chance — correction fixes this)
            p_bonf = min(u_p * n_comparisons, 1.0)  # cap at 1.0 because probability can't exceed 100%
            label = f"{CANCER_TYPES[i]} vs {CANCER_TYPES[j]}"
            stat_rows.append({
                'metric':      metric,
                'test':        'Mann-Whitney U (Bonferroni)',
                'comparison':  label,
                'statistic':   round(u_stat, 4),
                'p_value':     round(p_bonf, 6),
                'significant': p_bonf < 0.05,  # significant after correction = real difference
            })
            sig_flag = '*' if p_bonf < 0.05 else ''
            print(f"  {label}: U={u_stat:.1f}, p_bonf={p_bonf:.4f} {sig_flag}")

stats_df = pd.DataFrame(stat_rows)  # convert list of result dicts into a table
print("\nStatistical results summary:")
print(stats_df.to_string(index=False))  # print without row numbers

In [ ]:
# Cell 6 — Figure 4: Alpha diversity box plots
# Box plots show how diversity is distributed across samples in each cancer type
# The box covers the middle 50% of values; the line inside is the median; dots are individual samples

PALETTE = {
    'Colorectal': '#2196F3',  # blue for colorectal cancer
    'Breast':     '#E91E63',  # pink/red for breast cancer
    'Prostate':   '#4CAF50',  # green for prostate cancer
}

METRIC_LABELS = {
    'shannon':       'Shannon Entropy',    # human-readable axis label for each metric
    'observed_otus': 'Observed OTUs',
    'simpson':       'Simpson Index',
}

fig, axes = plt.subplots(1, 3, figsize=(14, 5))  # 3 side-by-side plots in one figure (1 row, 3 columns)
fig.suptitle('Alpha Diversity by Cancer Type', fontsize=14, fontweight='bold', y=1.02)  # overall figure title

# Build a lookup of Kruskal-Wallis p-values and significant pairwise comparisons for each metric
kw_pvals = {}  # stores the KW p-value per metric
pw_sig   = {}  # stores list of significant pairwise comparisons per metric
for metric in METRICS:
    sub     = stats_df[(stats_df['metric'] == metric)]  # filter to rows for this metric
    kw_row  = sub[sub['test'] == 'Kruskal-Wallis']      # get the overall KW test row
    kw_pvals[metric] = float(kw_row['p_value'].values[0]) if len(kw_row) else 1.0  # extract p-value
    pw_rows = sub[(sub['test'] == 'Mann-Whitney U (Bonferroni)') & (sub['significant'] == True)]
    pw_sig[metric] = list(pw_rows['comparison'].values)  # list of "A vs B" labels that were significant

CT_ORDER = ['Colorectal', 'Breast', 'Prostate']  # consistent left-to-right order on x-axis

for ax, metric in zip(axes, METRICS):
    # Draw box plot: boxes show quartile ranges, no outlier dots (handled by strip plot below)
    sns.boxplot(
        data=diversity_df,
        x='cancer_type', y=metric,
        order=CT_ORDER,
        palette=PALETTE,
        width=0.5,
        flierprops=dict(marker='', alpha=0),  # hide default outlier dots (strip plot will show all points)
        ax=ax,
    )
    # Overlay individual sample dots (jittered sideways so overlapping points separate)
    sns.stripplot(
        data=diversity_df,
        x='cancer_type', y=metric,
        order=CT_ORDER,
        palette=PALETTE,
        size=3, alpha=0.5, jitter=True,  # small semi-transparent dots, spread horizontally
        ax=ax,
    )

    ax.set_xlabel('Cancer Type', fontsize=11)    # x-axis label
    ax.set_ylabel(METRIC_LABELS[metric], fontsize=11)  # y-axis label
    ax.set_title(METRIC_LABELS[metric], fontsize=12)   # subplot title

    # Display the Kruskal-Wallis p-value in a box in the top-right corner of each subplot
    kw_p    = kw_pvals[metric]
    p_label = f"KW p = {kw_p:.4f}" if kw_p >= 0.0001 else "KW p < 0.0001"
    ax.text(
        0.97, 0.97, p_label,
        transform=ax.transAxes,      # position relative to subplot (0=left/bottom, 1=right/top)
        ha='right', va='top', fontsize=9,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.5),  # beige rounded box
    )

    # Draw significance brackets: horizontal lines above the plot connecting pairs that differ significantly
    y_vals   = diversity_df[metric].dropna()
    y_max    = y_vals.max()
    y_range  = y_vals.max() - y_vals.min()
    bracket_h   = y_range * 0.05  # height of the vertical tick at each end of the bracket
    bracket_top = y_max + y_range * 0.05  # starting y-position for first bracket

    x_map = {ct: i for i, ct in enumerate(CT_ORDER)}  # maps cancer type name → x position (0, 1, 2)
    bracket_level = 0  # stacks multiple brackets upward so they don't overlap
    for pair_label in pw_sig[metric]:
        parts = pair_label.split(' vs ')  # split "Breast vs Colorectal" into ["Breast", "Colorectal"]
        if len(parts) == 2 and parts[0] in x_map and parts[1] in x_map:
            x1, x2 = x_map[parts[0]], x_map[parts[1]]  # x positions of the two groups
            y = bracket_top + bracket_level * (y_range * 0.12)  # stagger each bracket higher
            ax.plot([x1, x1, x2, x2], [y, y + bracket_h, y + bracket_h, y], 'k-', lw=1.0)  # draw bracket
            ax.text((x1 + x2) / 2, y + bracket_h + bracket_h * 0.2, '*',
                    ha='center', va='bottom', fontsize=12)  # asterisk above bracket = significant
            bracket_level += 1  # next bracket goes higher

    ax.tick_params(axis='x', labelsize=10)  # slightly smaller x-axis tick labels

plt.tight_layout()  # automatically adjust spacing so subplots don't overlap
fig_path = os.path.join(FIGURES_DIR, 'fig04_alpha_diversity.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')  # save at high resolution (300 DPI = print quality)
plt.close()  # close figure to free memory
print(f"Saved: {fig_path}")

In [ ]:
# Cell 7 — Save results
# Save both output tables so other notebooks and the paper can use them

# Save the per-sample diversity metrics (Shannon, Observed OTUs, Simpson + labels)
diversity_out = os.path.join(RESULTS_DIR, 'alpha_diversity_metrics.csv')
diversity_df.to_csv(diversity_out)  # write table to CSV file (includes index = sample IDs)
print(f"Saved diversity metrics : {diversity_out}")

# Save the statistical test results (Kruskal-Wallis + pairwise Mann-Whitney with Bonferroni)
stats_out = os.path.join(RESULTS_DIR, 'alpha_diversity_stats.csv')
stats_df.to_csv(stats_out, index=False)  # index=False skips the 0,1,2,... row number column
print(f"Saved statistical tests : {stats_out}")

In [ ]:
# Cell 8 — Cancer vs Healthy comparison within Colorectal and Breast
# Question: within the same cancer type, is diversity different between cancer patients and healthy controls?
# Prostate has no healthy controls in this dataset, so we can only compare Colorectal and Breast

cancer_types_with_healthy = ['Colorectal', 'Breast']  # only these two have both groups

fig, axes = plt.subplots(1, 2, figsize=(10, 5))  # two side-by-side plots
fig.suptitle('Shannon Entropy: Cancer vs Healthy', fontsize=13, fontweight='bold')

condition_palette = {'Cancer': '#D32F2F', 'Healthy': '#388E3C'}  # red for cancer, green for healthy

for ax, ct in zip(axes, cancer_types_with_healthy):
    sub = diversity_df[diversity_df['cancer_type'] == ct].copy()  # keep only rows for this cancer type

    # Box plot comparing Cancer vs Healthy within this cancer type
    sns.boxplot(
        data=sub,
        x='condition', y='shannon',
        order=['Cancer', 'Healthy'],    # Cancer on left, Healthy on right
        palette=condition_palette,
        width=0.5,
        flierprops=dict(marker='', alpha=0),  # hide outlier dots (strip plot handles them)
        ax=ax,
    )
    # Overlay individual sample dots
    sns.stripplot(
        data=sub,
        x='condition', y='shannon',
        order=['Cancer', 'Healthy'],
        palette=condition_palette,
        size=3, alpha=0.5, jitter=True,
        ax=ax,
    )

    ax.set_title(f"{ct} Cancer", fontsize=12)          # e.g. "Colorectal Cancer"
    ax.set_xlabel('Condition', fontsize=11)
    ax.set_ylabel('Shannon Entropy', fontsize=11)

    # Mann-Whitney U test: compare Shannon entropy between cancer and healthy samples
    cancer_vals  = sub.loc[sub['condition'] == 'Cancer',  'shannon'].dropna().values
    healthy_vals = sub.loc[sub['condition'] == 'Healthy', 'shannon'].dropna().values

    if len(cancer_vals) > 0 and len(healthy_vals) > 0:
        u_stat, u_p = stats.mannwhitneyu(cancer_vals, healthy_vals, alternative='two-sided')
        p_label = f"MWU p = {u_p:.4f}" if u_p >= 0.0001 else "MWU p < 0.0001"  # MWU = Mann-Whitney U
        ax.text(
            0.97, 0.97, p_label,
            transform=ax.transAxes,    # position in top-right corner of subplot
            ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.5),
        )

        # If the test is significant, draw a significance bracket above the boxes
        if u_p < 0.05:
            y_max   = sub['shannon'].max()
            y_range = sub['shannon'].max() - sub['shannon'].min()
            bh = y_range * 0.05          # bracket tick height
            y  = y_max + y_range * 0.05  # bracket y-position (just above the highest value)
            ax.plot([0, 0, 1, 1], [y, y + bh, y + bh, y], 'k-', lw=1.0)  # bracket lines
            ax.text(0.5, y + bh + bh * 0.2, '*', ha='center', va='bottom', fontsize=12)  # significance star
    else:
        # If one group has no data, show a placeholder message
        ax.text(0.5, 0.5, 'Insufficient data', transform=ax.transAxes,
                ha='center', va='center', fontsize=10, color='gray')

plt.tight_layout()  # prevent subplots from overlapping
fig_path2 = os.path.join(FIGURES_DIR, 'fig04b_shannon_cancer_vs_healthy.png')
plt.savefig(fig_path2, dpi=300, bbox_inches='tight')  # save high-resolution chart
plt.close()  # close to free memory
print(f"Saved: {fig_path2}")

In [ ]:
# Cell 9 — Print summary table: mean ± std per cancer type
# A final human-readable report showing average diversity and spread (std) per cancer group
print("=" * 65)
print("  Alpha Diversity Summary: Mean ± Std by Cancer Type")
print("=" * 65)

# Build the header row for the table
header = f"{'Cancer Type':<14}  {'Shannon':>14}  {'Observed OTUs':>15}  {'Simpson':>12}"
print(header)
print("-" * 65)

for ct in CT_ORDER:  # loop through Colorectal, Breast, Prostate in order
    sub = diversity_df[diversity_df['cancer_type'] == ct]  # keep only rows for this cancer type
    n   = len(sub)  # number of samples in this group

    # Calculate mean and standard deviation for each diversity metric
    sh_mean, sh_std = sub['shannon'].mean(),       sub['shannon'].std()         # Shannon entropy
    ot_mean, ot_std = sub['observed_otus'].mean(), sub['observed_otus'].std()   # Observed OTUs
    si_mean, si_std = sub['simpson'].mean(),       sub['simpson'].std()          # Simpson index

    # Print one formatted row per cancer type (e.g., "Colorectal  2.609 ± 0.404")
    print(
        f"{ct:<14}  "                              # left-aligned cancer type name
        f"{sh_mean:>6.3f} ± {sh_std:<5.3f}  "     # Shannon: mean ± std
        f"{ot_mean:>6.1f} ± {ot_std:<6.1f}  "     # Observed OTUs: mean ± std
        f"{si_mean:>6.4f} ± {si_std:<6.4f}"       # Simpson: mean ± std
        f"  (n={n})"                               # sample count
    )

print("=" * 65)
print("\nNote: Prostate cancer has no healthy controls in this dataset.")
print("Notebook 03 complete.")